# Student Habits & Performance — Basic Dataset Info

Reading `student_habits_performance.csv` with pandas (only) and reporting basic information about the dataset.

In [1]:
import pandas as pd

df = pd.read_csv("student_habits_performance.csv")
df.head()

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,S1000,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,S1001,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,S1002,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,S1003,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,S1004,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


## Shape: number of rows and columns

In [2]:
n_rows, n_cols = df.shape
print(f"Number of rows:    {n_rows}")
print(f"Number of columns: {n_cols}")

Number of rows:    1000
Number of columns: 16


## Column names

In [3]:
print("Columns:")
for col in df.columns:
    print(f"  - {col}")

Columns:
  - student_id
  - age
  - gender
  - study_hours_per_day
  - social_media_hours
  - netflix_hours
  - part_time_job
  - attendance_percentage
  - sleep_hours
  - diet_quality
  - exercise_frequency
  - parental_education_level
  - internet_quality
  - mental_health_rating
  - extracurricular_participation
  - exam_score


## Column data types, non-null counts, and memory usage

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   student_id                     1000 non-null   str    
 1   age                            1000 non-null   int64  
 2   gender                         1000 non-null   str    
 3   study_hours_per_day            1000 non-null   float64
 4   social_media_hours             1000 non-null   float64
 5   netflix_hours                  1000 non-null   float64
 6   part_time_job                  1000 non-null   str    
 7   attendance_percentage          1000 non-null   float64
 8   sleep_hours                    1000 non-null   float64
 9   diet_quality                   1000 non-null   str    
 10  exercise_frequency             1000 non-null   int64  
 11  parental_education_level       909 non-null    str    
 12  internet_quality               1000 non-null   str    
 13  

## Missing values per column

In [5]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows:       {df.duplicated().sum()}")

Missing values per column:
student_id                        0
age                               0
gender                            0
study_hours_per_day               0
social_media_hours                0
netflix_hours                     0
part_time_job                     0
attendance_percentage             0
sleep_hours                       0
diet_quality                      0
exercise_frequency                0
parental_education_level         91
internet_quality                  0
mental_health_rating              0
extracurricular_participation     0
exam_score                        0
dtype: int64

Total missing values: 91
Duplicate rows:       0


## Summary statistics for numeric columns

In [6]:
df.describe()

,age,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,exercise_frequency,mental_health_rating,exam_score
count,1000.0000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.4980,3.55010,2.505500,1.819700,84.131700,6.470100,3.042000,5.438000,69.601500
std,2.3081,1.46889,1.172422,1.075118,9.399246,1.226377,2.025423,2.847501,16.888564
min,17.0000,0.00000,0.000000,0.000000,56.000000,3.200000,0.000000,1.000000,18.400000
25%,18.7500,2.60000,1.700000,1.000000,78.000000,5.600000,1.000000,3.000000,58.475000
50%,20.0000,3.50000,2.500000,1.800000,84.400000,6.500000,3.000000,5.000000,70.500000
75%,23.0000,4.50000,3.300000,2.525000,91.025000,7.300000,5.000000,8.000000,81.325000
max,24.0000,8.30000,7.200000,5.400000,100.000000,10.000000,6.000000,10.000000,100.000000


## Does discipline compensate for bad internet?

We proxy *discipline* with `study_hours_per_day`, split into Low / Medium / High
(by tertiles), and look at the average exam score for each combination of
internet quality and discipline level. If discipline compensates for bad
internet, the "Poor" internet row should still score well at "High" discipline.

In [7]:
df["discipline_level"] = pd.qcut(
    df["study_hours_per_day"],
    q=3,
    labels=["Low", "Medium", "High"],
)

internet_order = ["Poor", "Average", "Good"]
discipline_order = ["Low", "Medium", "High"]

avg_score_table = (
    df.pivot_table(
        index="internet_quality",
        columns="discipline_level",
        values="exam_score",
        aggfunc="mean",
        observed=True,
    )
    .reindex(index=internet_order, columns=discipline_order)
    .round(2)
)
print("Average exam score by internet quality x discipline (study hours):")
avg_score_table

Average exam score by internet quality x discipline (study hours):


discipline_level,Low,Medium,High
internet_quality,,,
Poor,54.79,73.08,83.33
Average,53.60,71.60,87.08
Good,53.89,68.89,83.43


## Average grade for different personas (feature combinations)

Each row is a category of people defined by a combination of input features.
We report how many students match and their average exam score.

In [8]:
personas = {
    "Sportsmen w/ good mental health":
        (df["exercise_frequency"] >= 5) & (df["mental_health_rating"] >= 7),
    "Part-time workers w/ educated parents":
        (df["part_time_job"] == "Yes") &
        (df["parental_education_level"].isin(["Bachelor", "Master"])),
    "Hard studiers, little social media":
        (df["study_hours_per_day"] >= 5) & (df["social_media_hours"] <= 1),
    "Night owls (little sleep, lots of Netflix)":
        (df["sleep_hours"] <= 5) & (df["netflix_hours"] >= 3),
    "Healthy lifestyle (sleep + diet + exercise)":
        (df["sleep_hours"] >= 7) & (df["diet_quality"] == "Good") &
        (df["exercise_frequency"] >= 4),
    "High attendance + good internet":
        (df["attendance_percentage"] >= 90) & (df["internet_quality"] == "Good"),
    "Distracted (high social media + high Netflix)":
        (df["social_media_hours"] >= 4) & (df["netflix_hours"] >= 3),
    "Active extracurriculars + studious":
        (df["extracurricular_participation"] == "Yes") & (df["study_hours_per_day"] >= 4),
    # --- additional combinations ---
    "Balanced achievers (study + sleep + low screen)":
        (df["study_hours_per_day"] >= 4) & (df["sleep_hours"] >= 7) &
        (df["social_media_hours"] <= 2) & (df["netflix_hours"] <= 2),
    "Burning the candle (high study, low sleep)":
        (df["study_hours_per_day"] >= 5) & (df["sleep_hours"] <= 5),
    "Working busy bees (job + extracurriculars)":
        (df["part_time_job"] == "Yes") & (df["extracurricular_participation"] == "Yes"),
    "Resilient grinders (studious despite bad mental health)":
        (df["study_hours_per_day"] >= 5) & (df["mental_health_rating"] <= 3),
    "Couch potatoes (no exercise + lots of screen time)":
        (df["exercise_frequency"] <= 1) & (df["social_media_hours"] >= 4) &
        (df["netflix_hours"] >= 2),
    "Low effort, high attendance (shows up but doesn't study)":
        (df["attendance_percentage"] >= 90) & (df["study_hours_per_day"] <= 2),
    "Thriving all-rounders (good mental health + diet + active)":
        (df["mental_health_rating"] >= 7) & (df["diet_quality"] == "Good") &
        (df["exercise_frequency"] >= 4),
    "Poor internet but disciplined":
        (df["internet_quality"] == "Poor") & (df["study_hours_per_day"] >= 5),
    "Younger students (<=19) who study hard":
        (df["age"] <= 19) & (df["study_hours_per_day"] >= 5),
    "Well-rested low-stress (good sleep + good mental health)":
        (df["sleep_hours"] >= 8) & (df["mental_health_rating"] >= 7),
}

persona_table = pd.DataFrame([
    {"persona": name, "count": int(mask.sum()),
     "avg_exam_score": round(df.loc[mask, "exam_score"].mean(), 2)}
    for name, mask in personas.items()
]).sort_values("avg_exam_score", ascending=False).reset_index(drop=True)

print(f"Overall average exam score: {df['exam_score'].mean():.2f}\n")
persona_table

Overall average exam score: 69.60



,persona,count,avg_exam_score
0,"Hard studiers, little social media",13,96.65
1,Younger students (<=19) who study hard,51,91.62
2,Balanced achievers (study + sleep + low screen),29,89.13
3,Poor internet but disciplined,27,85.36
4,"Burning the candle (high study, low sleep)",23,84.45
5,Active extracurriculars + studious,120,84.19
6,Resilient grinders (studious despite bad menta...,47,84.05
7,Sportsmen w/ good mental health,113,80.40
8,Thriving all-rounders (good mental health + di...,66,79.55
9,Well-rested low-stress (good sleep + good ment...,50,78.28


## Who gets good grades no matter the other factors?

Earlier analysis showed `study_hours_per_day` overwhelmingly dominates the grade.
Here we test the claim directly: take the **hard studiers** (top tertile of study
hours) and check their average exam score *within every level of every other
factor*. If the score stays high across all of them, then high study hours
"wins regardless of other factors". We compare against everyone else (the rest)
in the same subgroup.

In [9]:
hard_studier = df["discipline_level"] == "High"   # top tertile of study hours
print(f"Hard studiers: {hard_studier.sum()} | avg grade {df.loc[hard_studier,'exam_score'].mean():.2f} "
      f"vs rest {df.loc[~hard_studier,'exam_score'].mean():.2f}\n")

# Build comparison split factors: categoricals + binned numerics
split_factors = {
    "gender": df["gender"],
    "part_time_job": df["part_time_job"],
    "diet_quality": df["diet_quality"],
    "internet_quality": df["internet_quality"],
    "parental_education_level": df["parental_education_level"],
    "extracurricular_participation": df["extracurricular_participation"],
    "mental_health": pd.cut(df["mental_health_rating"], [0, 3, 6, 10],
                            labels=["Low", "Mid", "High"]),
    "sleep_hours": pd.cut(df["sleep_hours"], [0, 5, 7, 24],
                          labels=["<=5", "5-7", ">7"]),
    "exercise_frequency": pd.cut(df["exercise_frequency"], [-1, 1, 3, 6],
                                 labels=["Low", "Mid", "High"]),
    "social_media_hours": pd.cut(df["social_media_hours"], [-1, 2, 4, 24],
                                 labels=["Low", "Mid", "High"]),
}

rows = []
for fname, series in split_factors.items():
    for level, sub in df.groupby(series, observed=True).groups.items():
        sub_idx = df.index.isin(sub)
        hs = df[sub_idx & hard_studier.values]["exam_score"]
        rest = df[sub_idx & ~hard_studier.values]["exam_score"]
        rows.append({
            "factor": fname, "level": str(level),
            "hard_studier_avg": round(hs.mean(), 1) if len(hs) else None,
            "rest_avg": round(rest.mean(), 1) if len(rest) else None,
            "n_hard": len(hs),
        })

consistency_table = pd.DataFrame(rows)
print("Hard-studier avg grade within every level of every other factor:")
consistency_table

Hard studiers: 319 | avg grade 84.95 vs rest 62.41

Hard-studier avg grade within every level of every other factor:


,factor,level,hard_studier_avg,rest_avg,n_hard
0,gender,Female,84.7,62.3,160
1,gender,Male,85.3,62.3,146
2,gender,Other,83.5,64.9,13
3,part_time_job,No,85.4,62.6,249
4,part_time_job,Yes,83.3,61.7,70
5,diet_quality,Fair,84.9,62.8,151
6,diet_quality,Good,84.7,62.1,122
7,diet_quality,Poor,85.7,62.3,46
8,internet_quality,Average,87.1,62.1,134
9,internet_quality,Good,83.4,61.9,140


## Who benefits most from studying? (study-hours ↔ grade correlation by group)

For many different groups of people (defined by single features and feature
combinations) we compute the correlation between `study_hours_per_day` and
`exam_score`. A high positive correlation means that group's grade rises
steeply with study hours (they benefit most); a low or negative correlation
means extra study hours barely help — or even hurt — that group.

In [10]:
def study_corr(mask):
    sub = df[mask]
    if len(sub) < 10:           # skip tiny groups (unreliable correlation)
        return None
    return sub["study_hours_per_day"].corr(sub["exam_score"])

# Groups defined by single features and feature combinations
groups = {
    "ALL students": pd.Series(True, index=df.index),
    "Male": df["gender"] == "Male",
    "Female": df["gender"] == "Female",
    "Part-time job": df["part_time_job"] == "Yes",
    "No part-time job": df["part_time_job"] == "No",
    "Good mental health (>=7)": df["mental_health_rating"] >= 7,
    "Bad mental health (<=3)": df["mental_health_rating"] <= 3,
    "Good sleep (>=7h)": df["sleep_hours"] >= 7,
    "Poor sleep (<=5h)": df["sleep_hours"] <= 5,
    "Good diet": df["diet_quality"] == "Good",
    "Poor diet": df["diet_quality"] == "Poor",
    "Poor internet": df["internet_quality"] == "Poor",
    "Good internet": df["internet_quality"] == "Good",
    "High exercise (>=5)": df["exercise_frequency"] >= 5,
    "No exercise (<=1)": df["exercise_frequency"] <= 1,
    "Heavy social media (>=4h)": df["social_media_hours"] >= 4,
    "Light social media (<=1h)": df["social_media_hours"] <= 1,
    "Heavy Netflix (>=3h)": df["netflix_hours"] >= 3,
    "Extracurriculars": df["extracurricular_participation"] == "Yes",
    "Educated parents (Bachelor/Master)":
        df["parental_education_level"].isin(["Bachelor", "Master"]),
    # combinations
    "Bad mental health + part-time job":
        (df["mental_health_rating"] <= 3) & (df["part_time_job"] == "Yes"),
    "Good mental health + good sleep":
        (df["mental_health_rating"] >= 7) & (df["sleep_hours"] >= 7),
    "Poor sleep + heavy social media":
        (df["sleep_hours"] <= 5) & (df["social_media_hours"] >= 4),
    "Healthy (good sleep + diet + exercise)":
        (df["sleep_hours"] >= 7) & (df["diet_quality"] == "Good") &
        (df["exercise_frequency"] >= 4),
    "Distracted (heavy social media + Netflix)":
        (df["social_media_hours"] >= 4) & (df["netflix_hours"] >= 3),
    "Poor internet + part-time job":
        (df["internet_quality"] == "Poor") & (df["part_time_job"] == "Yes"),
}

study_corr_table = (
    pd.DataFrame([
        {"group": name, "n": int(mask.sum()),
         "study_grade_corr": (None if study_corr(mask) is None
                              else round(study_corr(mask), 3))}
        for name, mask in groups.items()
    ])
    .dropna(subset=["study_grade_corr"])
    .sort_values("study_grade_corr", ascending=False)
    .reset_index(drop=True)
)
print("Correlation between study hours and grade, per group "
      "(higher = benefits more from studying):")
study_corr_table

Correlation between study hours and grade, per group (higher = benefits more from studying):


,group,n,study_grade_corr
0,Distracted (heavy social media + Netflix),16,0.936
1,Heavy Netflix (>=3h),160,0.875
2,Good mental health + good sleep,137,0.869
3,Bad mental health (<=3),301,0.868
4,Healthy (good sleep + diet + exercise),69,0.859
5,High exercise (>=5),301,0.858
6,Good mental health (>=7),382,0.856
7,Poor diet,185,0.851
8,Good sleep (>=7h),345,0.839
9,No exercise (<=1),290,0.838


## Who benefits most from exercising? (exercise-frequency ↔ grade correlation by group)

Same idea as the study-hours analysis, but now correlating `exercise_frequency`
with `exam_score` within each group. A high positive correlation means that
group's grade rises with more exercise (they benefit most); a low or negative
value means exercise barely helps — or hurts — that group. Groups defined by
exercise itself are excluded (they'd be degenerate).

In [11]:
def exercise_corr(mask):
    sub = df[mask]
    if len(sub) < 10:           # skip tiny groups (unreliable correlation)
        return None
    return sub["exercise_frequency"].corr(sub["exam_score"])

# Reuse the same groups but drop the exercise-defined ones (degenerate here)
exercise_groups = {
    name: mask for name, mask in groups.items()
    if name not in ("High exercise (>=5)", "No exercise (<=1)",
                    "Healthy (good sleep + diet + exercise)")
}
# add a couple of study-related groups for contrast
exercise_groups["Hard studiers (>=5h)"] = df["study_hours_per_day"] >= 5
exercise_groups["Low studiers (<=2h)"] = df["study_hours_per_day"] <= 2

exercise_corr_table = (
    pd.DataFrame([
        {"group": name, "n": int(mask.sum()),
         "exercise_grade_corr": (None if exercise_corr(mask) is None
                                 else round(exercise_corr(mask), 3))}
        for name, mask in exercise_groups.items()
    ])
    .dropna(subset=["exercise_grade_corr"])
    .sort_values("exercise_grade_corr", ascending=False)
    .reset_index(drop=True)
)
print("Correlation between exercise frequency and grade, per group "
      "(higher = benefits more from exercising):")
exercise_corr_table

Correlation between exercise frequency and grade, per group (higher = benefits more from exercising):


,group,n,exercise_grade_corr
0,Hard studiers (>=5h),167,0.428
1,Low studiers (<=2h),154,0.285
2,Good mental health + good sleep,137,0.260
3,Poor internet,162,0.242
4,Bad mental health (<=3),301,0.220
5,No part-time job,785,0.192
6,Good mental health (>=7),382,0.186
7,Male,477,0.183
8,Good sleep (>=7h),345,0.172
9,ALL students,1000,0.160


## Features whose correlation with grade CHANGES SIGN across groups

Exercise was positive for most people but negative for the time-starved/distracted.
Here we search systematically: for every numeric feature, we compute its
correlation with `exam_score` within each group, and flag the features that are
positive for some types of people and negative for others (sign flip). For each
such feature we show the group where it helps most and the group where it hurts most.

In [12]:
candidate_features = [
    "study_hours_per_day", "social_media_hours", "netflix_hours",
    "attendance_percentage", "sleep_hours", "exercise_frequency",
    "mental_health_rating", "age",
]

# Compute corr(feature, exam_score) for every (feature, group), reusing `groups`.
records = []
for fname in candidate_features:
    for gname, mask in groups.items():
        sub = df[mask]
        if len(sub) < 20:                 # need a reasonable sample for a corr
            continue
        c = sub[fname].corr(sub["exam_score"])
        records.append({"feature": fname, "group": gname,
                        "n": len(sub), "corr": c})

corr_long = pd.DataFrame(records)

# A feature "changes sign" if it is meaningfully positive in some group
# and meaningfully negative in another (|corr| >= 0.05 on both sides).
THRESH = 0.05
summary = []
for fname, g in corr_long.groupby("feature"):
    pos = g[g["corr"] >= THRESH]
    neg = g[g["corr"] <= -THRESH]
    if len(pos) and len(neg):
        best = g.loc[g["corr"].idxmax()]
        worst = g.loc[g["corr"].idxmin()]
        summary.append({
            "feature": fname,
            "max_corr": round(best["corr"], 3),
            "helps_most_group": best["group"],
            "min_corr": round(worst["corr"], 3),
            "hurts_most_group": worst["group"],
            "spread": round(best["corr"] - worst["corr"], 3),
        })

sign_flip_table = (pd.DataFrame(summary)
                   .sort_values("spread", ascending=False)
                   .reset_index(drop=True))
print("Features that flip the sign of their correlation with grade "
      "across different types of people:\n")
sign_flip_table

Features that flip the sign of their correlation with grade across different types of people:



,feature,max_corr,helps_most_group,min_corr,hurts_most_group,spread
0,exercise_frequency,0.260,Good mental health + good sleep,-0.243,Poor internet + part-time job,0.503
1,sleep_hours,0.398,Poor internet + part-time job,-0.058,Heavy social media (>=4h),0.456
2,attendance_percentage,0.306,Poor internet + part-time job,-0.143,Poor internet,0.449
3,social_media_hours,0.061,Heavy social media (>=4h),-0.259,Poor internet + part-time job,0.320
4,age,0.205,Bad mental health + part-time job,-0.107,Good internet,0.311


## Sub-groups that shift the grade significantly within a factor level

For each factor level (the *baseline* — e.g. `internet_quality = Good`), we compute
its average grade. Then we split that baseline population by every other factor and
compute each sub-group's average grade. We keep only the rows where the sub-group's
average differs from the baseline by **more than 5 points** (absolute difference
≤ 5 is discarded). A positive `diff` means that sub-group raises the grade relative
to the baseline; negative means it lowers it.

In [13]:
# Categorical / binned views of every factor used both as baseline and as splitter
factor_views = {
    "gender": df["gender"],
    "part_time_job": df["part_time_job"],
    "diet_quality": df["diet_quality"],
    "internet_quality": df["internet_quality"],
    "parental_education_level": df["parental_education_level"],
    "extracurricular_participation": df["extracurricular_participation"],
    "study_hours": df["discipline_level"],                      # Low/Medium/High
    "mental_health": pd.cut(df["mental_health_rating"], [0, 3, 6, 10],
                            labels=["Low", "Mid", "High"]),
    "sleep_hours": pd.cut(df["sleep_hours"], [0, 5, 7, 24],
                          labels=["<=5", "5-7", ">7"]),
    "exercise_frequency": pd.cut(df["exercise_frequency"], [-1, 1, 3, 6],
                                 labels=["Low", "Mid", "High"]),
    "social_media_hours": pd.cut(df["social_media_hours"], [-1, 2, 4, 24],
                                 labels=["Low", "Mid", "High"]),
    "netflix_hours": pd.cut(df["netflix_hours"], [-1, 1, 3, 24],
                            labels=["Low", "Mid", "High"]),
}

MIN_DIFF = 5      # discard |diff| <= 5
MIN_N = 15        # ignore tiny, noisy sub-groups

rows = []
for base_name, base_series in factor_views.items():
    for base_level in base_series.dropna().unique():
        base_mask = base_series == base_level
        base_avg = df.loc[base_mask, "exam_score"].mean()
        for split_name, split_series in factor_views.items():
            if split_name == base_name:
                continue
            for split_level in split_series.dropna().unique():
                sub_mask = base_mask & (split_series == split_level)
                n = int(sub_mask.sum())
                if n < MIN_N:
                    continue
                group_avg = df.loc[sub_mask, "exam_score"].mean()
                diff = group_avg - base_avg
                if abs(diff) > MIN_DIFF:
                    rows.append({
                        "base_factor": f"{base_name}={base_level}",
                        "base_avg": round(base_avg, 1),
                        "split_group": f"{split_name}={split_level}",
                        "group_avg": round(group_avg, 1),
                        "diff": round(diff, 1),
                        "n": n,
                    })

significant_shifts = (
    pd.DataFrame(rows)
    .assign(abs_diff=lambda d: d["diff"].abs())
    .sort_values("abs_diff", ascending=False)
    .drop_duplicates(subset="split_group", keep="first")   # one row per split_group
    .drop(columns="abs_diff")
    .reset_index(drop=True)
)
print(f"Sub-groups shifting the grade by more than {MIN_DIFF} points "
      f"vs their factor baseline (n >= {MIN_N}), one row per split_group: "
      f"{len(significant_shifts)} rows\n")
significant_shifts

Sub-groups shifting the grade by more than 5 points vs their factor baseline (n >= 15), one row per split_group: 12 rows



,base_factor,base_avg,split_group,group_avg,diff,n
0,diet_quality=Poor,68.1,study_hours=High,85.7,17.6,46
1,social_media_hours=Mid,68.4,study_hours=Low,51.2,-17.2,179
2,exercise_frequency=Mid,69.3,netflix_hours=High,58.5,-10.8,30
3,parental_education_level=Master,68.1,sleep_hours=<=5,58.9,-9.2,20
4,netflix_hours=High,65.4,mental_health=Low,56.3,-9.1,36
5,diet_quality=Fair,70.4,social_media_hours=High,61.9,-8.6,35
6,social_media_hours=Mid,68.4,mental_health=High,76.5,8.1,201
7,sleep_hours=<=5,65.0,netflix_hours=Low,72.1,7.0,33
8,netflix_hours=High,65.4,exercise_frequency=Mid,58.5,-6.9,30
9,internet_quality=Poor,69.7,exercise_frequency=Low,63.3,-6.4,47


## Does bad mental health change the grade within each group?

For various feature combinations (none of which use mental health to define the
group), we compare the group's overall average grade against the average grade of
just the **bad mental health** students (`mental_health_rating <= 3`) inside that
same group. `diff` = bad-MH average − group average; a negative value means bad
mental health drags the grade down for that type of student.

In [14]:
# Groups defined WITHOUT mental health
mh_groups = {
    "Hard studiers (>=5h)": df["study_hours_per_day"] >= 5,
    "Low studiers (<=2h)": df["study_hours_per_day"] <= 2,
    "Part-time workers": df["part_time_job"] == "Yes",
    "Good sleep (>=7h)": df["sleep_hours"] >= 7,
    "Poor sleep (<=5h)": df["sleep_hours"] <= 5,
    "High exercise (>=5)": df["exercise_frequency"] >= 5,
    "No exercise (<=1)": df["exercise_frequency"] <= 1,
    "Heavy social media (>=4h)": df["social_media_hours"] >= 4,
    "Heavy Netflix (>=3h)": df["netflix_hours"] >= 3,
    "Poor internet": df["internet_quality"] == "Poor",
    "Good diet": df["diet_quality"] == "Good",
    "Poor diet": df["diet_quality"] == "Poor",
    "Extracurriculars": df["extracurricular_participation"] == "Yes",
    "Educated parents": df["parental_education_level"].isin(["Bachelor", "Master"]),
    "Studious + good sleep":
        (df["study_hours_per_day"] >= 5) & (df["sleep_hours"] >= 7),
    "Studious + part-time job":
        (df["study_hours_per_day"] >= 5) & (df["part_time_job"] == "Yes"),
    "Distracted (social media + Netflix)":
        (df["social_media_hours"] >= 4) & (df["netflix_hours"] >= 3),
    "Healthy (sleep + diet + exercise)":
        (df["sleep_hours"] >= 7) & (df["diet_quality"] == "Good") &
        (df["exercise_frequency"] >= 4),
}

bad_mh_mask = df["mental_health_rating"] <= 3

rows = []
for name, mask in mh_groups.items():
    grp = df.loc[mask, "exam_score"]
    bad = df.loc[mask & bad_mh_mask, "exam_score"]
    if len(bad) < 10:                 # need enough bad-MH students to be meaningful
        continue
    rows.append({
        "group": name,
        "group_avg": round(grp.mean(), 1),
        "bad_mh_avg": round(bad.mean(), 1),
        "diff": round(bad.mean() - grp.mean(), 1),
        "n_group": len(grp),
        "n_bad_mh": len(bad),
    })

bad_mh_effect = (pd.DataFrame(rows)
                 .sort_values("diff")
                 .reset_index(drop=True))
print("Effect of bad mental health on grade, within each group "
      "(diff = bad-MH avg - group avg):\n")
bad_mh_effect

Effect of bad mental health on grade, within each group (diff = bad-MH avg - group avg):



,group,group_avg,bad_mh_avg,diff,n_group,n_bad_mh
0,Studious + part-time job,89.0,79.9,-9.1,36,12
1,Poor diet,68.1,60.4,-7.7,185,53
2,Heavy Netflix (>=3h),65.9,58.6,-7.3,160,42
3,Good sleep (>=7h),72.1,65.1,-7.0,345,105
4,No exercise (<=1),66.1,59.1,-7.0,290,97
5,Poor sleep (<=5h),65.0,58.2,-6.8,127,44
6,Poor internet,69.7,63.3,-6.4,162,44
7,Hard studiers (>=5h),90.3,84.0,-6.3,167,47
8,Low studiers (<=2h),47.0,40.7,-6.2,154,37
9,Healthy (sleep + diet + exercise),73.5,67.3,-6.2,69,17


# Do these findings generalize to the larger (80k) dataset?

We re-run the key hypotheses above against the ~80k-row
`enhanced_student_habits_performance_dataset.csv`, loaded into a separate
`df_big` so the analyses above are untouched.

**Rules of this check:**
- Only columns shared with the original analysis are used; the new dataset's
  extra columns (major, previous_gpa, stress_level, motivation_level, ...) are ignored.
- Two harmonizations: `internet_quality` is `Low/Medium/High` here (mapped back to
  `Poor/Average/Good`) and `parental_education_level` has extra levels.
- The new dataset's exam scores skew much higher (mean ~89 vs ~70), so findings
  are judged by **relationships** (correlations, rankings, signs), not absolute grades.

In [15]:
df_big = pd.read_csv("enhanced_student_habits_performance_dataset.csv")

# Harmonize internet_quality to the original Poor/Average/Good naming
df_big["internet_quality"] = df_big["internet_quality"].map(
    {"Low": "Poor", "Medium": "Average", "High": "Good"})

# Discipline = study-hours tertiles, same construction as the original notebook
df_big["discipline_level"] = pd.qcut(df_big["study_hours_per_day"], q=3,
                                     labels=["Low", "Medium", "High"])

print(f"Rows: {len(df_big)}  |  overall avg exam score: {df_big['exam_score'].mean():.2f}")
df_big[["study_hours_per_day", "exam_score"]].describe().round(2)

Rows: 80000  |  overall avg exam score: 89.14


,study_hours_per_day,exam_score
count,80000.00,80000.00
mean,4.17,89.14
std,2.00,11.59
min,0.00,36.00
25%,2.80,82.00
50%,4.13,93.00
75%,5.50,100.00
max,12.00,100.00


### A. Does discipline compensate for bad internet? (same as small dataset)

In [16]:
internet_order = ["Poor", "Average", "Good"]
discipline_order = ["Low", "Medium", "High"]

avg_score_table_big = (
    df_big.pivot_table(
        index="internet_quality",
        columns="discipline_level",
        values="exam_score",
        aggfunc="mean",
        observed=True,
    )
    .reindex(index=internet_order, columns=discipline_order)
    .round(2)
)
print("Average exam score by internet quality x discipline (study hours):")
avg_score_table_big

Average exam score by internet quality x discipline (study hours):


discipline_level,Low,Medium,High
internet_quality,,,
Poor,86.27,89.37,92.11
Average,85.82,89.34,92.28
Good,86.08,89.33,92.09


### B. Average grade for different personas (same combinations as small dataset)

In [17]:
personas_big = {
    "Sportsmen w/ good mental health":
        (df_big["exercise_frequency"] >= 5) & (df_big["mental_health_rating"] >= 7),
    "Part-time workers w/ educated parents":
        (df_big["part_time_job"] == "Yes") &
        (df_big["parental_education_level"].isin(["Bachelor", "Master"])),
    "Hard studiers, little social media":
        (df_big["study_hours_per_day"] >= 5) & (df_big["social_media_hours"] <= 1),
    "Night owls (little sleep, lots of Netflix)":
        (df_big["sleep_hours"] <= 5) & (df_big["netflix_hours"] >= 3),
    "Healthy lifestyle (sleep + diet + exercise)":
        (df_big["sleep_hours"] >= 7) & (df_big["diet_quality"] == "Good") &
        (df_big["exercise_frequency"] >= 4),
    "High attendance + good internet":
        (df_big["attendance_percentage"] >= 90) & (df_big["internet_quality"] == "Good"),
    "Distracted (high social media + high Netflix)":
        (df_big["social_media_hours"] >= 4) & (df_big["netflix_hours"] >= 3),
    "Active extracurriculars + studious":
        (df_big["extracurricular_participation"] == "Yes") & (df_big["study_hours_per_day"] >= 4),
    # --- additional combinations ---
    "Balanced achievers (study + sleep + low screen)":
        (df_big["study_hours_per_day"] >= 4) & (df_big["sleep_hours"] >= 7) &
        (df_big["social_media_hours"] <= 2) & (df_big["netflix_hours"] <= 2),
    "Burning the candle (high study, low sleep)":
        (df_big["study_hours_per_day"] >= 5) & (df_big["sleep_hours"] <= 5),
    "Working busy bees (job + extracurriculars)":
        (df_big["part_time_job"] == "Yes") & (df_big["extracurricular_participation"] == "Yes"),
    "Resilient grinders (studious despite bad mental health)":
        (df_big["study_hours_per_day"] >= 5) & (df_big["mental_health_rating"] <= 3),
    "Couch potatoes (no exercise + lots of screen time)":
        (df_big["exercise_frequency"] <= 1) & (df_big["social_media_hours"] >= 4) &
        (df_big["netflix_hours"] >= 2),
    "Low effort, high attendance (shows up but doesn't study)":
        (df_big["attendance_percentage"] >= 90) & (df_big["study_hours_per_day"] <= 2),
    "Thriving all-rounders (good mental health + diet + active)":
        (df_big["mental_health_rating"] >= 7) & (df_big["diet_quality"] == "Good") &
        (df_big["exercise_frequency"] >= 4),
    "Poor internet but disciplined":
        (df_big["internet_quality"] == "Poor") & (df_big["study_hours_per_day"] >= 5),
    "Younger students (<=19) who study hard":
        (df_big["age"] <= 19) & (df_big["study_hours_per_day"] >= 5),
    "Well-rested low-stress (good sleep + good mental health)":
        (df_big["sleep_hours"] >= 8) & (df_big["mental_health_rating"] >= 7),
}

persona_table_big = pd.DataFrame([
    {"persona": name, "count": int(mask.sum()),
     "avg_exam_score": round(df_big.loc[mask, "exam_score"].mean(), 2)}
    for name, mask in personas_big.items()
]).sort_values("avg_exam_score", ascending=False).reset_index(drop=True)

print(f"Overall average exam score: {df_big['exam_score'].mean():.2f}\n")
persona_table_big

Overall average exam score: 89.14



,persona,count,avg_exam_score
0,Poor internet but disciplined,9131,92.08
1,"Hard studiers, little social media",5791,92.01
2,Younger students (<=19) who study hard,8529,91.97
3,Balanced achievers (study + sleep + low screen),4655,91.89
4,Active extracurriculars + studious,21512,91.31
5,Resilient grinders (studious despite bad menta...,847,91.26
6,Healthy lifestyle (sleep + diet + exercise),10471,90.77
7,"Burning the candle (high study, low sleep)",2669,90.67
8,Well-rested low-stress (good sleep + good ment...,10340,90.44
9,Sportsmen w/ good mental health,14438,90.37


### C. Who gets good grades no matter the other factors? (same as small dataset)

In [18]:
hard_studier_big = df_big["discipline_level"] == "High"   # top tertile of study hours
print(f"Hard studiers: {hard_studier_big.sum()} | avg grade {df_big.loc[hard_studier_big,'exam_score'].mean():.2f} "
      f"vs rest {df_big.loc[~hard_studier_big,'exam_score'].mean():.2f}\n")

# Build comparison split factors: categoricals + binned numerics
split_factors_big = {
    "gender": df_big["gender"],
    "part_time_job": df_big["part_time_job"],
    "diet_quality": df_big["diet_quality"],
    "internet_quality": df_big["internet_quality"],
    "parental_education_level": df_big["parental_education_level"],
    "extracurricular_participation": df_big["extracurricular_participation"],
    "mental_health": pd.cut(df_big["mental_health_rating"], [0, 3, 6, 10],
                            labels=["Low", "Mid", "High"]),
    "sleep_hours": pd.cut(df_big["sleep_hours"], [0, 5, 7, 24],
                          labels=["<=5", "5-7", ">7"]),
    "exercise_frequency": pd.cut(df_big["exercise_frequency"], [-1, 1, 3, 6],
                                 labels=["Low", "Mid", "High"]),
    "social_media_hours": pd.cut(df_big["social_media_hours"], [-1, 2, 4, 24],
                                 labels=["Low", "Mid", "High"]),
}

rows = []
for fname, series in split_factors_big.items():
    for level, sub in df_big.groupby(series, observed=True).groups.items():
        sub_idx = df_big.index.isin(sub)
        hs = df_big[sub_idx & hard_studier_big.values]["exam_score"]
        rest = df_big[sub_idx & ~hard_studier_big.values]["exam_score"]
        rows.append({
            "factor": fname, "level": str(level),
            "hard_studier_avg": round(hs.mean(), 1) if len(hs) else None,
            "rest_avg": round(rest.mean(), 1) if len(rest) else None,
            "n_hard": len(hs),
        })

consistency_table_big = pd.DataFrame(rows)
print("Hard-studier avg grade within every level of every other factor:")
consistency_table_big

Hard studiers: 26463 | avg grade 92.16 vs rest 87.65



Hard-studier avg grade within every level of every other factor:


,factor,level,hard_studier_avg,rest_avg,n_hard
0,gender,Female,92.1,87.6,8736
1,gender,Male,92.2,87.6,8863
2,gender,Other,92.2,87.7,8864
3,part_time_job,No,92.2,87.7,13384
4,part_time_job,Yes,92.2,87.6,13079
5,diet_quality,Fair,92.2,87.8,8752
6,diet_quality,Good,92.0,87.5,13318
7,diet_quality,Poor,92.4,87.6,4393
8,internet_quality,Average,92.3,87.5,8805
9,internet_quality,Good,92.1,87.7,8872


### D. Who benefits most from studying? (study-hours ↔ grade correlation by group)

In [19]:
def study_corr_big(mask):
    sub = df_big[mask]
    if len(sub) < 10:           # skip tiny groups (unreliable correlation)
        return None
    return sub["study_hours_per_day"].corr(sub["exam_score"])

# Groups defined by single features and feature combinations
groups_big = {
    "ALL students": pd.Series(True, index=df_big.index),
    "Male": df_big["gender"] == "Male",
    "Female": df_big["gender"] == "Female",
    "Part-time job": df_big["part_time_job"] == "Yes",
    "No part-time job": df_big["part_time_job"] == "No",
    "Good mental health (>=7)": df_big["mental_health_rating"] >= 7,
    "Bad mental health (<=3)": df_big["mental_health_rating"] <= 3,
    "Good sleep (>=7h)": df_big["sleep_hours"] >= 7,
    "Poor sleep (<=5h)": df_big["sleep_hours"] <= 5,
    "Good diet": df_big["diet_quality"] == "Good",
    "Poor diet": df_big["diet_quality"] == "Poor",
    "Poor internet": df_big["internet_quality"] == "Poor",
    "Good internet": df_big["internet_quality"] == "Good",
    "High exercise (>=5)": df_big["exercise_frequency"] >= 5,
    "No exercise (<=1)": df_big["exercise_frequency"] <= 1,
    "Heavy social media (>=4h)": df_big["social_media_hours"] >= 4,
    "Light social media (<=1h)": df_big["social_media_hours"] <= 1,
    "Heavy Netflix (>=3h)": df_big["netflix_hours"] >= 3,
    "Extracurriculars": df_big["extracurricular_participation"] == "Yes",
    "Educated parents (Bachelor/Master)":
        df_big["parental_education_level"].isin(["Bachelor", "Master"]),
    # combinations
    "Bad mental health + part-time job":
        (df_big["mental_health_rating"] <= 3) & (df_big["part_time_job"] == "Yes"),
    "Good mental health + good sleep":
        (df_big["mental_health_rating"] >= 7) & (df_big["sleep_hours"] >= 7),
    "Poor sleep + heavy social media":
        (df_big["sleep_hours"] <= 5) & (df_big["social_media_hours"] >= 4),
    "Healthy (good sleep + diet + exercise)":
        (df_big["sleep_hours"] >= 7) & (df_big["diet_quality"] == "Good") &
        (df_big["exercise_frequency"] >= 4),
    "Distracted (heavy social media + Netflix)":
        (df_big["social_media_hours"] >= 4) & (df_big["netflix_hours"] >= 3),
    "Poor internet + part-time job":
        (df_big["internet_quality"] == "Poor") & (df_big["part_time_job"] == "Yes"),
}

study_corr_table_big = (
    pd.DataFrame([
        {"group": name, "n": int(mask.sum()),
         "study_grade_corr": (None if study_corr_big(mask) is None
                              else round(study_corr_big(mask), 3))}
        for name, mask in groups_big.items()
    ])
    .dropna(subset=["study_grade_corr"])
    .sort_values("study_grade_corr", ascending=False)
    .reset_index(drop=True)
)
print("Correlation between study hours and grade, per group "
      "(higher = benefits more from studying):")
study_corr_table_big

Correlation between study hours and grade, per group (higher = benefits more from studying):


,group,n,study_grade_corr
0,Distracted (heavy social media + Netflix),4381,0.270
1,Poor sleep (<=5h),7667,0.255
2,Educated parents (Bachelor/Master),31894,0.253
3,Poor diet,13352,0.248
4,Heavy social media (>=4h),16882,0.247
5,Heavy Netflix (>=3h),21002,0.246
6,No exercise (<=1),19833,0.244
7,Male,26698,0.243
8,Healthy (good sleep + diet + exercise),10471,0.243
9,Good sleep (>=7h),41069,0.243


### E. Who benefits most from exercising? (exercise-frequency ↔ grade correlation by group)

In [20]:
def exercise_corr_big(mask):
    sub = df_big[mask]
    if len(sub) < 10:           # skip tiny groups (unreliable correlation)
        return None
    return sub["exercise_frequency"].corr(sub["exam_score"])

# Reuse the same groups but drop the exercise-defined ones (degenerate here)
exercise_groups_big = {
    name: mask for name, mask in groups_big.items()
    if name not in ("High exercise (>=5)", "No exercise (<=1)",
                    "Healthy (good sleep + diet + exercise)")
}
# add a couple of study-related groups for contrast
exercise_groups_big["Hard studiers (>=5h)"] = df_big["study_hours_per_day"] >= 5
exercise_groups_big["Low studiers (<=2h)"] = df_big["study_hours_per_day"] <= 2

exercise_corr_table_big = (
    pd.DataFrame([
        {"group": name, "n": int(mask.sum()),
         "exercise_grade_corr": (None if exercise_corr_big(mask) is None
                                 else round(exercise_corr_big(mask), 3))}
        for name, mask in exercise_groups_big.items()
    ])
    .dropna(subset=["exercise_grade_corr"])
    .sort_values("exercise_grade_corr", ascending=False)
    .reset_index(drop=True)
)
print("Correlation between exercise frequency and grade, per group "
      "(higher = benefits more from exercising):")
exercise_corr_table_big

Correlation between exercise frequency and grade, per group (higher = benefits more from exercising):


,group,n,exercise_grade_corr
0,Poor sleep + heavy social media,1662,0.111
1,Distracted (heavy social media + Netflix),4381,0.105
2,Poor sleep (<=5h),7667,0.103
3,Bad mental health + part-time job,1221,0.101
4,Heavy social media (>=4h),16882,0.096
5,Male,26698,0.095
6,Good mental health (>=7),38581,0.093
7,Heavy Netflix (>=3h),21002,0.092
8,Good mental health + good sleep,19914,0.092
9,Poor internet,26714,0.090


### F. Features whose correlation with grade CHANGES SIGN across groups (same as small dataset)

In [21]:
candidate_features = [
    "study_hours_per_day", "social_media_hours", "netflix_hours",
    "attendance_percentage", "sleep_hours", "exercise_frequency",
    "mental_health_rating", "age",
]

# Compute corr(feature, exam_score) for every (feature, group), reusing `groups_big`.
records = []
for fname in candidate_features:
    for gname, mask in groups_big.items():
        sub = df_big[mask]
        if len(sub) < 20:                 # need a reasonable sample for a corr
            continue
        c = sub[fname].corr(sub["exam_score"])
        records.append({"feature": fname, "group": gname,
                        "n": len(sub), "corr": c})

corr_long_big = pd.DataFrame(records)

# A feature "changes sign" if it is meaningfully positive in some group
# and meaningfully negative in another (|corr| >= 0.05 on both sides).
THRESH = 0.05
summary = []
for fname, g in corr_long_big.groupby("feature"):
    pos = g[g["corr"] >= THRESH]
    neg = g[g["corr"] <= -THRESH]
    if len(pos) and len(neg):
        best = g.loc[g["corr"].idxmax()]
        worst = g.loc[g["corr"].idxmin()]
        summary.append({
            "feature": fname,
            "max_corr": round(best["corr"], 3),
            "helps_most_group": best["group"],
            "min_corr": round(worst["corr"], 3),
            "hurts_most_group": worst["group"],
            "spread": round(best["corr"] - worst["corr"], 3),
        })

cols = ["feature", "max_corr", "helps_most_group", "min_corr", "hurts_most_group", "spread"]
sign_flip_table_big = (pd.DataFrame(summary, columns=cols)
                       .sort_values("spread", ascending=False)
                       .reset_index(drop=True))
print("Features that flip the sign of their correlation with grade "
      "across different types of people:\n")
if sign_flip_table_big.empty:
    print("(none — no feature changes sign across groups in the large dataset)")
sign_flip_table_big

Features that flip the sign of their correlation with grade across different types of people:

(none — no feature changes sign across groups in the large dataset)


,feature,max_corr,helps_most_group,min_corr,hurts_most_group,spread


### G. Sub-groups that shift the grade significantly within a factor level (same as small dataset)

In [22]:
# Categorical / binned views of every factor used both as baseline and as splitter
factor_views_big = {
    "gender": df_big["gender"],
    "part_time_job": df_big["part_time_job"],
    "diet_quality": df_big["diet_quality"],
    "internet_quality": df_big["internet_quality"],
    "parental_education_level": df_big["parental_education_level"],
    "extracurricular_participation": df_big["extracurricular_participation"],
    "study_hours": df_big["discipline_level"],                  # Low/Medium/High
    "mental_health": pd.cut(df_big["mental_health_rating"], [0, 3, 6, 10],
                            labels=["Low", "Mid", "High"]),
    "sleep_hours": pd.cut(df_big["sleep_hours"], [0, 5, 7, 24],
                          labels=["<=5", "5-7", ">7"]),
    "exercise_frequency": pd.cut(df_big["exercise_frequency"], [-1, 1, 3, 6],
                                 labels=["Low", "Mid", "High"]),
    "social_media_hours": pd.cut(df_big["social_media_hours"], [-1, 2, 4, 24],
                                 labels=["Low", "Mid", "High"]),
    "netflix_hours": pd.cut(df_big["netflix_hours"], [-1, 1, 3, 24],
                            labels=["Low", "Mid", "High"]),
}

MIN_DIFF = 5      # discard |diff| <= 5
MIN_N = 15        # ignore tiny, noisy sub-groups

rows = []
for base_name, base_series in factor_views_big.items():
    for base_level in base_series.dropna().unique():
        base_mask = base_series == base_level
        base_avg = df_big.loc[base_mask, "exam_score"].mean()
        for split_name, split_series in factor_views_big.items():
            if split_name == base_name:
                continue
            for split_level in split_series.dropna().unique():
                sub_mask = base_mask & (split_series == split_level)
                n = int(sub_mask.sum())
                if n < MIN_N:
                    continue
                group_avg = df_big.loc[sub_mask, "exam_score"].mean()
                diff = group_avg - base_avg
                if abs(diff) > MIN_DIFF:
                    rows.append({
                        "base_factor": f"{base_name}={base_level}",
                        "base_avg": round(base_avg, 1),
                        "split_group": f"{split_name}={split_level}",
                        "group_avg": round(group_avg, 1),
                        "diff": round(diff, 1),
                        "n": n,
                    })

significant_shifts_big = (
    pd.DataFrame(rows)
    .assign(abs_diff=lambda d: d["diff"].abs())
    .sort_values("abs_diff", ascending=False)
    .drop_duplicates(subset="split_group", keep="first")   # one row per split_group
    .drop(columns="abs_diff")
    .reset_index(drop=True)
) if rows else pd.DataFrame(
    columns=["base_factor", "base_avg", "split_group", "group_avg", "diff", "n"])
print(f"Sub-groups shifting the grade by more than {MIN_DIFF} points "
      f"vs their factor baseline (n >= {MIN_N}), one row per split_group: "
      f"{len(significant_shifts_big)} rows\n")
significant_shifts_big

Sub-groups shifting the grade by more than 5 points vs their factor baseline (n >= 15), one row per split_group: 0 rows



,base_factor,base_avg,split_group,group_avg,diff,n


### H. Does bad mental health change the grade within each group? (same as small dataset)

In [23]:
# Groups defined WITHOUT mental health
mh_groups_big = {
    "Hard studiers (>=5h)": df_big["study_hours_per_day"] >= 5,
    "Low studiers (<=2h)": df_big["study_hours_per_day"] <= 2,
    "Part-time workers": df_big["part_time_job"] == "Yes",
    "Good sleep (>=7h)": df_big["sleep_hours"] >= 7,
    "Poor sleep (<=5h)": df_big["sleep_hours"] <= 5,
    "High exercise (>=5)": df_big["exercise_frequency"] >= 5,
    "No exercise (<=1)": df_big["exercise_frequency"] <= 1,
    "Heavy social media (>=4h)": df_big["social_media_hours"] >= 4,
    "Heavy Netflix (>=3h)": df_big["netflix_hours"] >= 3,
    "Poor internet": df_big["internet_quality"] == "Poor",
    "Good diet": df_big["diet_quality"] == "Good",
    "Poor diet": df_big["diet_quality"] == "Poor",
    "Extracurriculars": df_big["extracurricular_participation"] == "Yes",
    "Educated parents": df_big["parental_education_level"].isin(["Bachelor", "Master"]),
    "Studious + good sleep":
        (df_big["study_hours_per_day"] >= 5) & (df_big["sleep_hours"] >= 7),
    "Studious + part-time job":
        (df_big["study_hours_per_day"] >= 5) & (df_big["part_time_job"] == "Yes"),
    "Distracted (social media + Netflix)":
        (df_big["social_media_hours"] >= 4) & (df_big["netflix_hours"] >= 3),
    "Healthy (sleep + diet + exercise)":
        (df_big["sleep_hours"] >= 7) & (df_big["diet_quality"] == "Good") &
        (df_big["exercise_frequency"] >= 4),
}

bad_mh_mask_big = df_big["mental_health_rating"] <= 3

rows = []
for name, mask in mh_groups_big.items():
    grp = df_big.loc[mask, "exam_score"]
    bad = df_big.loc[mask & bad_mh_mask_big, "exam_score"]
    if len(bad) < 10:                 # need enough bad-MH students to be meaningful
        continue
    rows.append({
        "group": name,
        "group_avg": round(grp.mean(), 1),
        "bad_mh_avg": round(bad.mean(), 1),
        "diff": round(bad.mean() - grp.mean(), 1),
        "n_group": len(grp),
        "n_bad_mh": len(bad),
    })

bad_mh_effect_big = (pd.DataFrame(rows)
                     .sort_values("diff")
                     .reset_index(drop=True))
print("Effect of bad mental health on grade, within each group "
      "(diff = bad-MH avg - group avg):\n")
bad_mh_effect_big

Effect of bad mental health on grade, within each group (diff = bad-MH avg - group avg):



,group,group_avg,bad_mh_avg,diff,n_group,n_bad_mh
0,Healthy (sleep + diet + exercise),90.8,88.8,-2.0,10471,317
1,Studious + part-time job,92.1,90.5,-1.6,13642,436
2,Distracted (social media + Netflix),89.0,87.6,-1.5,4381,109
3,Good diet,89.0,87.6,-1.5,39935,1221
4,Poor diet,89.2,87.9,-1.3,13352,416
5,Studious + good sleep,92.8,91.7,-1.2,14184,451
6,Part-time workers,89.1,88.0,-1.1,39805,1221
7,Extracurriculars,89.2,88.1,-1.1,39942,1237
8,Low studiers (<=2h),84.4,83.4,-1.0,11939,394
9,Good sleep (>=7h),90.0,89.0,-0.9,41069,1262


## Verdict: do the small-dataset findings generalize to the 80k dataset?

Sections A–H above re-run the **exact same code/definitions** as the corresponding
small-dataset analyses, so this is a direct comparison.

| | Analysis | Small dataset | Large dataset (80k) | Holds? |
|---|---|---|---|---|
| A | Discipline compensates for bad internet | Low≈54 → High≈84, internet rows ~equal | Low≈86 → High≈92, internet rows ~equal | ✅ holds strongly |
| B | Persona ranking | studious personas top (~85–97), low-effort bottom (47) | studious personas top (~91–92), low-effort bottom (84) | ✅ ranking holds (compressed) |
| C | Hard studiers win regardless of other factors | min hard ~78 > non-hard max ~69 | min hard 90.8 > non-hard max 88.6 — beat rest in **every** cell | ✅ holds strongly |
| D | Study hours help every group | all +; distracted/poor-sleep benefit most | all +0.21–0.27; distracted/poor-sleep benefit most | ✅ holds (magnitude ~⅓) |
| E | Exercise weak & sometimes negative | weak; negative for time-starved/distracted | weak +0.08–0.11; **never negative** | ⚠️ weakness holds, sign-flip gone |
| F | Features that change sign across groups | 5 features flip (exercise, sleep, attendance, age, social media) | **none** — every feature keeps one consistent sign | ❌ does not reproduce |
| G | Sub-groups shifting grade > 5 pts | 12 rows (study hours swings ±17) | **0 rows** — no sub-group moves its baseline by > 5 pts | ❌ effect sizes fall below the 5-pt threshold |
| H | Bad mental health lowers grade in every group | every diff negative, −4 to −9 | every diff negative, −0.5 to −2.0 | ⚠️ direction holds, much smaller |

**Conclusion.** The *directional* story is robust at 80k rows: study discipline is the
dominant lever (A, B, C, D), hard studiers beat their peers in literally every
subgroup, and bad mental health is a small universal drag (H). What does **not**
survive is anything that depended on large effect *magnitudes* or on small, noisy
subgroups in the original 1k sample: the exercise sign-flip and the broader sign-flip
finding (F), and the >5-point sub-group shifts (G), all vanish because the 80k
dataset's grades are much higher and compressed (mean 89, sd ~12) and every
correlation is roughly a third of its original size. Net: trust the rankings and
directions; do not trust the original effect sizes or the fragile sign-flips.